In [ ]:
import pandas as pd
import requests
import time
import io
import re
from google.colab import files

print("Sube tu archivo CSV:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

try:
    df = pd.read_csv(io.BytesIO(uploaded[filename]), sep=None, engine='python')
except Exception as e:
    print(f"Error al leer el archivo: {e}")

cols = {col.lower(): col for col in df.columns}
nombre_col = cols.get('nombre')
autor_col = cols.get('autor')

def limpiar_texto(texto):
    if not isinstance(texto, str): return ""
    return re.split(r'[:\-\(]', texto)[0].strip()

# --- FUENTE 1: GOOGLE BOOKS ---
def buscar_google(titulo, autor):
    q = f"intitle:{titulo}+inauthor:{autor}"
    url = f"https://www.googleapis.com/books/v1/volumes?q={q}&maxResults=3"
    try:
        res = requests.get(url, timeout=5).json()
        if "items" in res:
            for item in res["items"]:
                img = item["volumeInfo"].get("imageLinks", {}).get("thumbnail")
                if img: return img.replace("http://", "https://")
    except:
        pass
    return None

# --- FUENTE 2: APPLE BOOKS ---
def buscar_apple(titulo, autor):
    url = f"https://itunes.apple.com/search?term={titulo}+{autor}&entity=ebook&limit=1"
    try:
        res = requests.get(url, timeout=5).json()
        if res.get("resultCount", 0) > 0:
            img = res["results"][0].get("artworkUrl100")
            # Truco para pedir la imagen en mejor calidad (600px en vez de 100px)
            if img: return img.replace("100x100bb", "600x600bb")
    except:
        pass
    return None

# --- FUENTE 3: OPEN LIBRARY ---
def buscar_open_library(titulo, autor):
    url = f"https://openlibrary.org/search.json?title={titulo}&author={autor}&limit=1"
    try:
        res = requests.get(url, timeout=5).json()
        if res.get("numFound", 0) > 0:
            cover_id = res["docs"][0].get("cover_i")
            if cover_id:
                return f"https://covers.openlibrary.org/b/id/{cover_id}-L.jpg"
    except:
        pass
    return None

# --- EL CEREBRO MULTI-FUENTE ---
def buscar_portada_definitiva(titulo, autor):
    t_limpio = limpiar_texto(titulo)
    a_limpio = limpiar_texto(autor)
    
    # Intento 1: Google
    url = buscar_google(t_limpio, a_limpio)
    if url: return url
    
    # Intento 2: Apple Books
    url = buscar_apple(t_limpio, a_limpio)
    if url: return url
    
    # Intento 3: Open Library
    url = buscar_open_library(t_limpio, a_limpio)
    if url: return url
    
    return "No encontrada"

if not nombre_col or not autor_col:
    print("No se encontraron las columnas 'Nombre' y 'Autor'.")
else:
    print(f"\nProcesando {len(df)} libros con múltiples APIs...")
    urls = []
    
    for i, row in df.iterrows():
        t = row[nombre_col]
        a = row[autor_col]
        print(f"Buscando [{i+1}/{len(df)}]: {t}")
        url = buscar_portada_definitiva(t, a)
        urls.append(url)
        time.sleep(0.3) # Respetar límites de los servidores

    df['URL_Foto'] = urls
    output_name = "catalogo_multifuente.csv"
    df.to_csv(output_name, index=False, encoding='utf-8-sig')
    files.download(output_name)
    print("\n✅ ¡Listo! Descargando el catálogo con la máxima cobertura posible.")